In [ ]:
import pandas as pd

pd.set_option("future.no_silent_downcasting", True)

def clean_data(settlements):
    
    settlements = settlements.rename(columns={"admin_cgaz_1": "province"})
    settlements = settlements.rename(columns={"admin_cgaz_2": "district"})
   
    settlements.loc[settlements["province"].str.lower() == "Nothern Cape".lower(), "province"] = "Northern Cape"
    
    def parse_fatalities(series, prefix):
        split = series.str.split(";", expand = True).apply(pd.to_numeric, errors = "coerce").fillna(0).astype(int)
        names = ["battles", "violence_civilian", "riots", "explosions"]
        
        for i, n in enumerate(names):
            settlements[f"{prefix}_{n}"] = split.iloc[:, i]
        settlements[f"{prefix}_total"] = split.sum(axis = 1)
        
    parse_fatalities(settlements["fatalities_within_25km"], "fat25km")
    parse_fatalities(settlements["fatalities_within_50km"], "fat50km")
    
    settlements = settlements.drop(columns = ["fatalities_within_25km", "fatalities_within_50km"])
    
    settlements = settlements.fillna({"mean_rwi": 0})
    settlements = settlements.fillna({"nearest_hub_name": "Unknown"})
    
    settlements["building_density_percent"] = settlements["building_density_percent"].clip(upper = 100.0)
    outlier_cols = ["demand", "population", "num_buildings", "hull_area",
                    "num_connections", "very_small_structures",
                    "large_buildings", "medium_buildings", "small_buildings"]
    p99 = settlements[outlier_cols].quantile(0.99)
    settlements["is_urban_outlier"] = (settlements[outlier_cols] > p99).any(axis = 1).astype(int)
    
    for col in outlier_cols:
        cap = settlements[col].quantile(0.99)
        capped = (settlements[col] > cap).sum()
        settlements[col] = settlements[col].clip(upper = cap).astype(settlements[col].dtype)
        if capped > 0:
            print(f"{col}: capped {capped} rows at {cap:,.1f}")
            
    bool_cols = ["main_road_access", "has_education_facility", "has_health_facility", "has_nightlight"]
    settlements[bool_cols] = settlements[bool_cols].astype(int)
    settlements["security_risk_score"] = settlements["security_risk"].map({"low": 0, "medium": 1, "high": 2})
    settlements = settlements.drop(columns = ["security_risk"])
    threshold = settlements["demand_connection"].quantile(0.25)
    settlements["is_underserved"] = (settlements["demand_connection"] <= threshold).astype(int)
    return settlements


settlements = pd.read_csv("south_africa_dre_atlas_settlements.csv")
settlements_clean = clean_data(settlements.copy())
settlements_clean.to_csv("settlements_clean.csv", index = False)
print(settlements_clean.head())